## 메타데이터 정보 조회

> `@/docs/10-FCONLINE/14-Meta-Data.md`

### 개요

`GET /static/fconline/meta/spid.json` 으로 전체 선수 목록을 수집한다.
인증 불필요. 총 **85,338명** 수록.

API 응답의 `id` 필드는 9자리 정수이며, 구조는 다음과 같다.

| 구분 | 타입 | 설명 |
| --- | --- | --- |
| `ID` | int | 선수 고유 식별자 (spid 원본 9자리) |
| `season_ID` | int | 시즌(클래스) 식별자 — 앞 3자리 |
| `player_ID` | int | 선수 식별자 — 뒤 6자리 |
| `name` | str | 선수명 |

### ERD

```mermaid
erDiagram
    meta_spid {
        INT ID PK
        INT season_ID
        INT player_ID
        STRING name
    }
```

### 저장 위치

```
@data/meta_spid.csv
```

In [3]:
import csv
import os

import requests
from dotenv import load_dotenv

load_dotenv()

_BASE_URL = os.getenv("SERVERS")
_SPID_URL = f"{_BASE_URL}/static/fconline/meta/spid.json"

OUTPUT = "../data/meta_spid.csv"


def fetch_spid_meta() -> list[dict]:
    resp = requests.get(_SPID_URL)
    resp.raise_for_status()
    return resp.json()


def parse_spid(raw: list[dict]) -> list[dict]:
    result = []
    for item in raw:
        spid = item["id"]
        spid_str = str(spid)
        result.append({
            "ID": spid,
            "season_ID": int(spid_str[:3]),
            "player_ID": int(spid_str[3:]),
            "name": item["name"],
        })
    return result


def save_csv(records: list[dict], path: str) -> None:
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=["ID", "season_ID", "player_ID", "name"])
        writer.writeheader()
        writer.writerows(records)
    print(f"저장 완료: {path} ({len(records)}건)")


if __name__ == "__main__":
    print("spid 메타데이터 수집 중...")
    raw = fetch_spid_meta()
    records = parse_spid(raw)
    print(f"총 {len(records)}명")
    save_csv(records, OUTPUT)


python-dotenv could not parse statement starting at line 15


spid 메타데이터 수집 중...
총 85338명
저장 완료: ../data/meta_spid.csv (85338건)
